# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [7]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [8]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['EPUMJUNWEW', 'IZXTZJFCZO'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 5, 16, 21, 13, 10, 21, 14, 23,  5, 23],
       [ 9, 26, 24, 20, 26, 10,  6,  3, 26, 15]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 23,  5, 23, 14, 21, 10, 13, 21, 16],
       [ 0, 15, 26,  3,  6, 10, 26, 20, 24, 26]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[23,  5, 23, 14, 21, 10, 13, 21, 16,  5],
       [15, 26,  3,  6, 10, 26, 20, 24, 26,  9]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [9]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    def call(self, enc_ids, dec_ids):
        '''
        带 attention 机制的 sequence2sequence 模型。
        使用双线性(bilinear) attention: score = h_dec @ W @ h_enc^T
        '''
        # --- Encoder ---
        enc_emb = self.embed_layer(enc_ids)           # (b, enc_len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)    # enc_out: (b, enc_len, h)

        # --- Decoder (teacher forcing) ---
        dec_emb = self.embed_layer(dec_ids)           # (b, dec_len, emb_sz)
        dec_out, _ = self.decoder(dec_emb, initial_state=[enc_state])  # (b, dec_len, h)

        # --- Bilinear Attention ---
        # enc_proj = W * enc_out => (b, enc_len, h)
        enc_proj = self.dense_attn(enc_out)
        # scores: (b, dec_len, h) x (b, h, enc_len) -> (b, dec_len, enc_len)
        scores = tf.matmul(dec_out, enc_proj, transpose_b=True)
        attn_weights = tf.nn.softmax(scores, axis=-1)  # (b, dec_len, enc_len)

        # context: weighted sum of encoder outputs => (b, dec_len, h)
        context = tf.matmul(attn_weights, enc_out)

        # 拼接 decoder 输出与 context 后预测
        combined = tf.concat([dec_out, context], axis=-1)  # (b, dec_len, 2h)
        logits = self.dense(combined)                       # (b, dec_len, v_sz)
        return logits
    
    
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        单步解码，带 bilinear attention。
        x:       (b_sz,)           当前输入 token
        state:   decoder cell 隐状态列表
        enc_out: (b_sz, enc_len, h) encoder 所有时刻输出
        '''
        inp_emb = self.embed_layer(x)                   # (b, emb_sz)
        h, new_state = self.decoder_cell(inp_emb, state)  # h: (b, h)

        # Bilinear attention
        enc_proj = self.dense_attn(enc_out)             # (b, enc_len, h)
        h_expand = tf.expand_dims(h, axis=1)            # (b, 1, h)
        scores = tf.matmul(h_expand, enc_proj, transpose_b=True)  # (b, 1, enc_len)
        attn_weights = tf.nn.softmax(scores, axis=-1)   # (b, 1, enc_len)
        context = tf.matmul(attn_weights, enc_out)      # (b, 1, h)
        context = tf.squeeze(context, axis=1)           # (b, h)

        combined = tf.concat([h, context], axis=-1)     # (b, 2h)
        logits = self.dense(combined)                   # (b, v_sz)
        out = tf.argmax(logits, axis=-1, output_type=tf.int32)
        return out, new_state

# Loss函数以及训练逻辑

In [10]:
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [11]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.2946446
step 500 : loss 1.3608114
step 1000 : loss 0.23757295
step 1500 : loss 0.100789405


<tf.Tensor: shape=(), dtype=float32, numpy=0.04574386402964592>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [12]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, False, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, False, True, True, True, True, True, True, True]
[('YOMXOFXHGYVKPMHEAWGO', 'OGWAEHMPKVYGHXFOXMOY'), ('XBGJHDXBUIFCYDOVBGKE', 'EKGBVODYCFIUBXDHJGBX'), ('QNQBSHNDQCMIHPYZXNZS', 'SZNXZYPHIMCQDNHSBQLQ'), ('BBZQHLPWCTNEQWVGJDDF', 'FDDJGVWQENTCWPLHQZBB'), ('TODTOEJNRQKXDXIWMVYF', 'FYVMWIXDXKQRNJEOTDRT'), ('CQSUMQERMYATFXYTUABI', 'IBAUTYXFTAYMREQMUSQC'), ('YAMAJZXGHDJFAQAJAMQJ', 'ADMAJAQAFJDHGXZWUYRY'), ('ZOMOQASCBIZIRUPXSNNP', 'PNNSXPURIZIBCSAQOMOZ'), ('IBIBIFIYYLLBITWGRIIM', 'MIIRJGWTIBLYYIFIBIBK'), ('XHDQTPFTUBXHDFHQYJYM', 'MYJYQHFDHXBUTFPTDBIL'), ('GKGQRRZOKLZOLPOWREHV', 'VHERWOPLOZLKOZRRQGKI'), ('RUVDYFVLWMXGCGHMDEWZ', 'ZWEDMHGCGXMWLVFYDVUR'), ('DQYDIJEQDRXHFSWEZUWQ', 'QWUZEWSFHXRDQEJIDYQD'), ('ZCJWRCZDXHLVHUXFTUJG', 'GJUTFXUHVLHXDZCRWJCZ'), ('LOMLXKUNJUYETAZTAYBK', 'KBYATZATEYUJNUKXLMOL'), ('VZRXTWOXVESBGYWEWUBI', 'IBUWEWYGBSEVXOWTXRZV'), ('